In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

windowed_df = pd.read_csv('../output/windown_sliced_data/windowed_df_10d.csv', parse_dates=['date', 'window_start', 'window_end', 'test_date'])
windowed_df

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
0,2019-04-28,16f4b,0.651879,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
1,2019-04-29,16f4b,0.704693,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
2,2019-04-30,16f4b,0.603180,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
3,2019-05-01,16f4b,0.651631,0.0,1.200000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
4,2019-05-02,16f4b,0.668107,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
...,...,...,...,...,...,...,...,...,...,...,...
5407,2019-06-21,ec812,0.661363,0.0,1.125000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5408,2019-06-22,ec812,0.623560,0.0,0.666667,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5409,2019-06-23,ec812,0.708682,0.0,0.875000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5410,2019-06-24,ec812,0.656330,0.0,0.888889,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False


In [2]:
if_results = pd.read_csv("../output/Anomaly_delirium_Revised/FI_anomaly_labels.csv")
if_results

,patient_id,window_start,test_date,anomaly_score,anomaly_label
0,16f4b,2019-04-28,2019-05-08,0.085888,1
1,16f4b,2019-05-03,2019-05-13,0.100597,1
2,1fbe4,2019-04-25,2019-05-05,0.079158,1
3,1fbe4,2019-04-26,2019-05-06,0.056187,1
4,1fbe4,2019-04-27,2019-05-07,0.008614,1
...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.067374,1
488,ec812,2019-06-12,2019-06-22,-0.011182,-1
489,ec812,2019-06-13,2019-06-23,0.046199,1
490,ec812,2019-06-14,2019-06-24,0.119338,1


In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

PREDICTORS = ['entropy_rate', 'early_warning_score',
              'sleep_quality_score', 'agitation_counts', 'uti_happen']
N_PERMUTATIONS = 20

# Identify anomaly events (anomaly_label == -1)
anomaly_events = if_results[if_results['anomaly_label'] == -1][
    ['patient_id', 'window_start', 'test_date']
].reset_index(drop=True)

fi_records = []

n_count = 0
for _, row in anomaly_events.iterrows():
    n_count= n_count+1
    print(n_count)
    pid = row['patient_id']
    ws  = row['window_start']
    td  = row['test_date']

    # Retrieve the exact window used during IF scoring
    window_data = windowed_df[
        (windowed_df['patient_id'] == pid) &
        (windowed_df['window_start'] == ws)
        ]

    baseline  = window_data[window_data['is_test_day'] == False][PREDICTORS].values
    test_row  = window_data[window_data['is_test_day'] == True][PREDICTORS].values

    if len(test_row) == 0:
        continue

    # Refit IF on this window's baseline (identical config to original run)
    clf = IsolationForest(
        n_estimators=20,
        max_samples=10,
        contamination=0.1,
        random_state=42
    )
    clf.fit(baseline)

    # Original anomaly score for the test day
    original_score = clf.decision_function(test_row)[0]

    for feat_idx, feat_name in enumerate(PREDICTORS):
        permuted_scores = []
        for _ in range(N_PERMUTATIONS):
            # Shuffle the 10 baseline values of feature A, keep all others unchanged
            permuted_baseline = baseline.copy()
            permuted_baseline[:, feat_idx] = np.random.permutation(baseline[:, feat_idx])

            # Refit on the permuted baseline
            clf_perm = IsolationForest(
                n_estimators=20,
                max_samples=10,
                contamination=0.1,
                random_state=42
            )
            clf_perm.fit(permuted_baseline)

            # Score the original unmodified test row
            permuted_scores.append(clf_perm.decision_function(test_row)[0])

        importance = original_score - np.mean(permuted_scores)

        fi_records.append({
            'patient_id':   pid,
            'window_start': ws,
            'anomaly_date': td,       # test_date of the anomaly event
            'feature':      feat_name,
            'importance':   importance
        })

fi_df = pd.DataFrame(fi_records)
fi_df.to_csv("../output/Anomaly_delirium_Revised/FI_permutation_importance.csv", index=False)
print(fi_df.head(10))
print(f"\nTotal records: {len(fi_df)}  "
      f"({len(anomaly_events)} anomaly events × {len(PREDICTORS)} features)")

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
  patient_id window_start anomaly_date              feature  importance
0      1fbe4   2019-04-28   2019-05-08         entropy_rate   -0.018133
1      1fbe4   2019-04-28   2019-05-08  early_warning_score   -0.001232
2      1fbe4   2019-04-28   2019-05-08  sleep_quality_score   -0.004112
3      1fbe4   2019-04-28   2019-05-08     agitation_counts    0.000000
4      1fbe4   2019-04-28   2019-05-08           uti_happen   -0.015323
5      1fbe4   2019-05-01   2019-05-11         entropy_rate   -0.002462
6      1fbe4   2019-05-01   2019-05-11  early_warning_score    0.019617
7      1fbe4   2019-05-01   2019-05-11  sleep_quality_score    0.003398
8      1fbe4   2019-05-01   2019-05-11     agitation_counts    0.000000
9      1fbe4   2019-05-01   2019-05-11           uti_happe

In [4]:
anomaly_events.patient_id.unique()

<StringArray>
['1fbe4', '30a32', '55cd4', '93c14', '96adf', 'a2849', 'c55f8', 'c5785',
 'c8574', 'e2472', 'ec812']
Length: 11, dtype: str